In [ ]:
import pandas as pd
import numpy as np

DATA = '../data/case-study/processed'

# gaze stability threshold
gaze_threshold = 0.01

for s in [0, 1, 2, 3]:
    filename = f'{DATA}/sed_{s:02d}.csv' if s > 0 else f'{DATA}/sed.csv'
    sed_df = pd.read_csv(filename)

    # rename columns
    sed_df = sed_df.rename(columns={
        'gazeDir.x': 'gaze_x',
        'gazeDir.y': 'gaze_y',
        'gazeDir.z': 'gaze_z'
    })

    # validate data
    null_mask = sed_df[['reltime', 'gaze_x', 'gaze_y', 'gaze_z']].isnull().any(axis=1)
    if null_mask.any():
        print(f"WARNING: Session {s}: dropping {null_mask.sum()} rows with missing values.")
        sed_df = sed_df.dropna(subset=['reltime', 'gaze_x', 'gaze_y', 'gaze_z'])

    # gaze direction diff
    sed_df['gaze_diff'] = np.sqrt((sed_df['gaze_x'].diff() ** 2) +
                                  (sed_df['gaze_y'].diff() ** 2) +
                                  (sed_df['gaze_z'].diff() ** 2))

    # identify fixation points
    sed_df['fixation'] = sed_df['gaze_diff'] < gaze_threshold

    # label fixation groups
    sed_df['fixation_id'] = (sed_df['fixation'] != sed_df['fixation'].shift()).cumsum()

    # filter fixation points
    fixation_df = sed_df[sed_df['fixation']]

    # fixation duration
    fixation_duration = fixation_df.groupby('fixation_id')['reltime'].agg(['min', 'max'])
    fixation_duration['duration'] = fixation_duration['max'] - fixation_duration['min']
    fixation_duration = fixation_duration[['duration']]

    # merge durations back
    sed_df = sed_df.merge(fixation_duration, left_on='fixation_id', right_index=True, how='left')

    # save data
    output_path = f'{DATA}/sed_fix_{s:02d}.csv' if s > 0 else f'{DATA}/sed_fix.csv'
    sed_df.to_csv(output_path, index=False)
    label = f"Session {s}" if s > 0 else "Baseline"
    print(f"{label}: saved to {output_path}")